# 한강공원 주차장 혼잡도 예측 — 베이스라인

이 노트북은 **데이터 준비까지는 완성되어 있고, 모델링 부분은 직접 채워야 합니다.**

**완성되어 있는 부분 (그대로 실행)**
- DB 조회 → DataFrame 생성
- 이상치 제거 + 기본 전처리
- Train / Validation / Test 3분할
- 기본 모델(LinearRegression) 학습 → 평가 → pkl 저장까지 1회 완주

**직접 채워야 하는 부분 (빈 셀 + 가이드)**
- 여러 모델 비교
- 하이퍼파라미터 튜닝 (Optuna 등)
- 앙상블 (Voting, Stacking 등)

## Train / Test 데이터는 "기간이 달라야" 합니다

캐글 대회에서 train.csv(타깃 포함)와 test.csv(타깃 미포함)를 따로 주는 이유는, **test 데이터가 학습 시점에 존재하지 않았던 미래 데이터**이기 때문입니다. 학습에 쓴 데이터와 같은 기간의 데이터를 test로 쓰면, 모델이 이미 비슷한 패턴을 본 상태에서 평가받는 셈이라 성능이 과장됩니다.

이 프로젝트도 마찬가지입니다. `use_date` 기준으로 가장 과거 60%를 Train, 그다음 20%를 Validation, 가장 최근 20%를 Test로 나눕니다. Test는 "학습 시점 이후 실제로 일어날 미래"를 흉내 낸 것이라고 생각하세요. 절대 무작위로 섞어서 나누면 안 됩니다.

각 빈 셀 위 마크다운에 "실전 프로젝트에서는 이런 작업을 한다"는 힌트만 적어뒀습니다. 정답 코드는 없습니다 — 수업 시간에 배운 자전거 수요예측, 캐글 주택가격 실습 패턴을 떠올리면서 직접 작성하세요.

실행 순서: 위에서 아래로 순서대로 셀을 실행하세요 (Shift + Enter)

## 1. 환경 설정 (완성)

원본 베이스라인의 3가지 항목을 그대로 따릅니다.
- 필수 라이브러리 호출
- 데이터 로드 및 메모리 최적화 작업 (DB 조회 단계에서 수행)
- random_seed 고정

```bash
pip install pandas scikit-learn joblib matplotlib seaborn sqlalchemy pymysql python-dotenv
```

In [1]:
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.rcParams["axes.unicode_minus"] = False #그래프에서 음수 부호(-)가 깨지는 문제 방지

SEED = 42 #난수 시드 고정 : 실행할 때마다 같은 결과가 나오도록 재현성 확보
np.random.seed(SEED)

print("라이브러리 로드 완료")

라이브러리 로드 완료


## 2. DB 연결 + 데이터 조회 (완성)

In [2]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD  = os.getenv("DB_PASSWORD")
DB_HOST      = os.getenv("DB_HOST")
DB_PORT      = os.getenv("DB_PORT")
DB_NAME      = os.getenv("DB_NAME")
DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
with engine.connect() as conn:
    print("DB 연결 성공" if conn.execute(text("SELECT 1")).scalar() == 1 else "DB 연결 실패")

DB 연결 성공


In [3]:
sql = """
    SELECT
        d.lot_id,
        i.capacity,
        MONTH(d.use_date)                                      AS month,
        DAYOFWEEK(d.use_date)                                  AS day_of_week,
        CASE WHEN DAYOFWEEK(d.use_date) IN (1, 7)
            THEN 1 ELSE 0
        END AS is_weekend,
        WEEKOFYEAR(d.use_date)                                 AS week_of_year,
        CASE WHEN h.holiday_date IS NOT NULL THEN 1 ELSE 0 END AS is_holiday,
        d.use_date,
        ROUND(d.daily_count / i.capacity * 100, 2)             AS occupancy_pct
    FROM parking_daily AS d
    INNER JOIN parking_info AS i ON d.lot_id = i.id
    LEFT JOIN holidays AS h ON d.use_date = h.holiday_date
    WHERE d.daily_count > 0 AND i.capacity > 0
    ORDER BY d.use_date, d.lot_id
"""

df = pd.read_sql(text(sql), engine)
print(f"로드 완료: {len(df):,}행 x {len(df.columns)}열")
df.head()

로드 완료: 21,880행 x 9열


,lot_id,capacity,month,day_of_week,is_weekend,week_of_year,is_holiday,use_date,occupancy_pct
0,2,1782,1,3,0,3,0,2020-01-14,0.11
1,2,1782,1,4,0,3,0,2020-01-15,6.40
2,7,99,1,4,0,3,0,2020-01-15,12.12
3,2,1782,1,5,0,3,0,2020-01-16,48.99
4,7,99,1,5,0,3,0,2020-01-16,28.28


## 3. 기본 전처리 — 이상치 제거 (완성)

`daily_count`가 `capacity`를 초과해서 혼잡도가 100%를 넘는 행은 데이터 오류로 보고 제거합니다.

원본 베이스라인의 "피처 엔지니어링(결측치 처리·범주형 처리·시간 데이터 분리)" 항목 중 결측치 처리에 해당합니다. 범주형 처리(`lot_id`는 이미 정수 ID)와 시간 데이터 분리(`month`, `day_of_week` 등)는 다음 SQL 조회 단계에서 이미 끝나 있습니다.

## 4. EDA (선택)

원본 베이스라인이 안내하는 EDA 체크리스트를 그대로 따르면 됩니다.
- `df.info()`, `df.describe()`, `df.isnull().sum()`
- 타겟(`occupancy_pct`) 분포 확인 시각화
- 피처들 분포 확인 시각화

이 섹션은 모델 성능에 직접 영향을 주지 않으므로 시간이 부족하면 건너뛰어도 됩니다.

In [4]:
# 자유롭게 EDA 코드를 작성하세요.
# 힌트: df.info(), df.describe(), df.groupby("lot_id")["occupancy_pct"].mean() 등


## 5. 피처(X) / 타깃(Y) 분리 (완성)

`FEATURE_COLUMNS`의 순서는 `predict.py`에서 예측할 때도 반드시 동일해야 합니다. 임의로 순서를 바꾸지 마세요.

In [5]:
FEATURE_COLUMNS = [
    "lot_id", "capacity", "month", "day_of_week",
    "is_weekend", "week_of_year", "is_holiday"
]

TARGET_COLUMN = "occupancy_pct"

#시간 순서 유지(시계열 데이터는 무작위로 섞으면 미래 정보가 학습에 새어 들어갈 수 있음)
df_sorted = df.sort_values("use_date").reset_index(drop=True)

X = df_sorted[FEATURE_COLUMNS]
y = df_sorted[TARGET_COLUMN]

print(f"X shape:{X.shape}");
print(f"y shape:{y.shape}");
X.head()

X shape:(21880, 7)
y shape:(21880,)


,lot_id,capacity,month,day_of_week,is_weekend,week_of_year,is_holiday
0,2,1782,1,3,0,3,0
1,2,1782,1,4,0,3,0
2,7,99,1,4,0,3,0
3,2,1782,1,5,0,3,0
4,7,99,1,5,0,3,0


## 6. Train / Validation / Test 3분할 (완성)

```
전체 데이터
├── Train      (60%) → 모델 학습
├── Validation (20%) → 모델 비교 / 하이퍼파라미터 튜닝 시 성능 확인
└── Test       (20%) → 모든 작업이 끝난 뒤 딱 한 번만 사용하는 최종 평가
```

**왜 Validation이 필요한가**: 모델 1개만 기본값으로 학습한다면 Train/Test 2분할로 충분합니다. 하지만 ① 여러 모델을 비교하거나 ② 하이퍼파라미터를 튜닝하려면, 그 과정에서 성능을 확인할 데이터가 Test셋과 분리되어 있어야 합니다. Test셋을 튜닝 중에 들여다보면 "시험 문제를 미리 본 것"과 같아서 최종 평가가 과장됩니다.

시간 순서를 유지하기 위해 `train_test_split()`의 무작위 섞기 대신 날짜 기준으로 순서대로 자릅니다.

## 7. 평가 지표 함수 (완성)

수업에서 배운 RMSLE / RMSE / MAE를 그대로 사용합니다.

In [6]:
def rmsle(y, pred):
    """로그 변환 후 RMSE. 작은 값과 큰 값의 상대 오차를 동등하게 평가합니다."""
    log_y    = np.log1p(y)
    log_pred = np.log1p(np.maximum(pred, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))


def rmse(y, pred):
    return np.sqrt(mean_squared_error(y, pred))


def evaluate_regr(y, pred, name=""):
    """RMSLE / RMSE / MAE 세 지표를 한 번에 출력합니다."""
    rmsle_val = rmsle(y, pred)
    rmse_val  = rmse(y, pred)
    mae_val   = mean_absolute_error(y, pred)
    label = f"[{name}] " if name else ""
    print(f"{label}RMSLE: {rmsle_val:.4f}, RMSE: {rmse_val:.3f}, MAE: {mae_val:.3f}")
    return {"name": name, "rmsle": rmsle_val, "rmse": rmse_val, "mae": mae_val}

## 8. 기본 모델 학습 + 평가 (완성)

원본 베이스라인은 "하이퍼파라미터 튜닝 없이 기본값으로만 LightGBM이나 XGBoost를 학습"하라고 안내합니다. 다만 이 노트북에서는 가장 단순한 `LinearRegression`을 1차 기준선으로 사용합니다. 그 이유는 다음과 같습니다.

- 기준선은 "이보다는 무조건 잘해야 한다"는 가장 낮은 허들 역할이면 충분합니다.
- LightGBM/XGBoost 자체를 9단계(여러 모델 비교)에서 직접 학생이 추가하도록 남겨두기 위함입니다.

**여기서는 LinearRegression으로 한 번 완주**하고, 9단계에서 LightGBM·XGBoost를 포함해 여러 모델을 직접 비교하세요.

## 9. 여러 모델 비교 (직접 작성)

기본 모델(LinearRegression) 하나만으로는 성능이 충분한지 알 수 없습니다. 8단계에서 나온 RMSLE 값이 바로 **기준선(baseline) 점수**입니다. 원본 베이스라인의 문구를 그대로 인용하면:

> "RMSLE값이 나오면 이후로 이 값을 개선시키는 코드를 완성해야 한다."

지금부터의 모든 단계(9~12단계)는 8단계의 RMSLE보다 더 낮은 값을 만드는 것이 목표입니다. **Validation셋**을 기준으로 여러 모델을 비교해서 어떤 알고리즘이 이 데이터에 더 잘 맞는지 확인하세요.

실전 프로젝트에서 보통 이렇게 진행합니다:

1. `Ridge`, `Lasso` 같은 정규화 선형 모델 추가
2. `RandomForestRegressor`, `GradientBoostingRegressor` 같은 트리 앙상블 추가
3. `XGBRegressor`, `LGBMRegressor` 같은 부스팅 모델 추가 (`pip install xgboost lightgbm` 필요, 원본 베이스라인이 추천하는 모델)
4. 각 모델을 `X_train, y_train`으로 학습 → `X_val, y_val`로 평가
5. `evaluate_regr()` 함수를 그대로 재사용해서 결과를 비교

**주의**: 이 단계에서는 절대 `X_test, y_test`를 사용하지 마세요. Test셋은 마지막 한 번만 사용합니다.

In [7]:
# 직접 작성하세요.
# 힌트:
#   from sklearn.linear_model import Ridge, Lasso
#   from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
#   각 모델을 X_train, y_train으로 학습 -> X_val, y_val로 evaluate_regr() 호출
#   결과를 리스트나 DataFrame으로 모아서 비교하면 보기 좋습니다.


In [8]:
# 모델 비교 결과를 표나 그래프로 정리하세요.
# 힌트: pd.DataFrame(results_list).sort_values("rmsle")


## 10. 하이퍼파라미터 튜닝 (직접 작성)

9단계에서 가장 성능이 좋았던 모델 1~2개를 골라 하이퍼파라미터를 튜닝하세요.

실전 프로젝트에서 보통 이렇게 진행합니다:

1. 튜닝할 파라미터가 1~2개로 적은 선형 모델(Ridge, Lasso) → `GridSearchCV`로 격자 탐색
2. 튜닝할 파라미터가 많은 트리/부스팅 모델(RandomForest, LightGBM, XGBoost) → `Optuna`로 자동 탐색 (`pip install optuna` 필요)
3. 교차검증은 `TimeSeriesSplit`을 사용해서 시간 순서를 유지하세요 (수업에서 배운 패턴)
4. 평가 지표(RMSLE)를 직접 스코어러로 만들어서 사용하세요 (`make_scorer()`)
5. 튜닝이 끝나면 **Validation셋**으로 다시 한번 성능을 확인하세요

**주의**: 여기서도 `X_test, y_test`는 사용하지 마세요.

In [9]:
# 직접 작성하세요.
# 힌트:
#   from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
#   from sklearn.metrics import make_scorer
#   import optuna
#
#   GridSearchCV 사용 시:
#     params = {"alpha": [...]}
#     grid = GridSearchCV(model, params, scoring=원하는_스코어러, cv=TimeSeriesSplit(n_splits=5))
#
#   Optuna 사용 시:
#     def objective(trial):
#         params = {"n_estimators": trial.suggest_int(...), ...}
#         ...
#         return 평가지표
#     study = optuna.create_study(direction="minimize")
#     study.optimize(objective, n_trials=50)


In [10]:
# 튜닝된 모델을 Validation셋으로 평가하세요.


## 11. 앙상블 (직접 작성)

튜닝된 모델 여러 개를 묶어서 단일 모델보다 더 안정적인 예측을 만들 수 있는지 확인하세요.

실전 프로젝트에서 보통 이렇게 진행합니다:

1. **Voting**: 여러 모델의 예측값을 (가중)평균 — `sklearn.ensemble.VotingRegressor`
2. **Stacking**: 베이스 모델들의 예측값을 메타모델이 다시 학습 — `sklearn.ensemble.StackingRegressor`, 또는 수업에서 배운 K-Fold OOF 수동 구현
3. Voting과 Stacking 중 어느 쪽이 이 데이터에 더 잘 맞는지 **Validation셋** 성능으로 비교

**주의**: 마지막 12단계 전까지는 `X_test, y_test`를 사용하지 마세요.

In [11]:
# 직접 작성하세요.
# 힌트:
#   from sklearn.ensemble import VotingRegressor, StackingRegressor
#   from sklearn.linear_model import Ridge
#   from sklearn.model_selection import KFold
#
#   VotingRegressor(estimators=[("이름1", 모델1), ("이름2", 모델2)], weights=[...])
#   StackingRegressor(estimators=[...], final_estimator=Ridge())


In [12]:
# Voting과 Stacking을 Validation셋으로 비교하세요.


## 12. 최종 모델 선택 (직접 작성)

9~11단계에서 만든 모든 모델 중 **Validation셋 성능이 가장 좋은 모델**을 최종 모델로 선택하세요. 선택 기준은 RMSLE가 가장 낮은 모델입니다.

선택한 모델을 `final_model` 이라는 변수명으로 만들어두면 이후 13단계 평가/저장 셀을 그대로 사용할 수 있습니다.

### Test셋은 "실제 수능 시험지"입니다

원본 베이스라인의 비유를 그대로 가져오면, Test셋은 실제 수능 시험지와 같습니다. 모든 모의고사(Validation 평가)가 끝나고 가장 자신 있는 모델 하나를 정한 뒤, **단 한 번만** 응시하는 것입니다. Test셋 점수를 보고 다시 모델을 고치러 9~11단계로 돌아가면 안 됩니다 — 그 순간부터는 더 이상 정직한 평가가 아닙니다.

### 컬럼 순서 경고

원본 베이스라인의 경고를 그대로 인용합니다.

> "학습할 때 사용한 X_train의 컬럼 순서와 예측할 때 넣는 X_test의 컬럼 순서가 완벽히 일치해야 한다."

이 프로젝트에서는 `FEATURE_COLUMNS` 상수(5단계에서 정의)가 그 순서를 강제합니다. `routers/predict.py`에서 예측할 때도 반드시 이 순서와 동일한 순서로 피처를 구성해야 합니다.

In [13]:
# 최종 모델을 final_model 변수에 할당하세요.
# final_model = ...


## 13. Test셋 최종 평가 (완성)

지금까지 한 번도 사용하지 않은 **Test셋**으로 `final_model`을 딱 한 번 평가합니다. 이 점수가 실제 배포했을 때 기대할 수 있는 성능입니다.

## 14. pkl 저장 (완성)

## 15. 저장된 모델 검증 (완성)

## 완료!

`predict.py`는 `joblib.load()`로 이 pkl을 그대로 읽어서 `model.predict(X)`를 호출하는 구조이므로, 9~12단계에서 어떤 모델/앙상블을 선택하든 `final_model`에 scikit-learn 호환 `.predict()` 인터페이스를 가진 객체만 들어 있으면 별도 코드 수정이 필요 없습니다.

(단, 로그 변환된 타깃으로 학습했다면 `predict.py`에서 `np.expm1()` 역변환을 추가해야 합니다 — 9~11단계에서 로그 변환을 사용했다면 잊지 마세요.)

서버 재시작 후 Swagger에서 `POST /predict`로 확인하세요.

```bash
python main.py
```